<a href="https://colab.research.google.com/github/silvia-j-escobar/ExternDataScience/blob/main/Task_Run_3_RAG_Configurations_and_Log_Output_Differences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook helps you experiment with Top-K retrieval and Similarity Thresholds using LlamaIndex and HuggingFace Embeddings.

Silvia Escobar


# ✅ STEP 1: Setup

In [1]:
!pip install llama-index pymupdf llama-index-embeddings-huggingface



In [3]:
from llama_index.core import VectorStoreIndex, Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.settings import Settings
import fitz  # PyMuPDF
import time

In [4]:
from google.colab import files
uploaded = files.upload()

Saving sample_contract.pdf to sample_contract.pdf


In [5]:
# Load a sample PDF or plain text (You can upload your own contract PDF here)
pdf_path = "/content/sample_contract.pdf"
doc = fitz.open(pdf_path)
text = "\\n".join([page.get_text() for page in doc])
#documents = [Document(text=text)]

In [6]:
# Configure the embedding model
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.embed_model = embed_model

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
# Define a sentence splitter (can also use TokenTextSplitter or CharacterTextSplitter)
text_splitter = SentenceSplitter(chunk_size=50, chunk_overlap=50)

# Turn raw text into a list of Document objects
documents = [Document(text=text)]

# Convert into nodes (smaller chunks)
nodes = text_splitter.get_nodes_from_documents(documents)

# Then create the index from these nodes
index = VectorStoreIndex(nodes)


# ✅ STEP 2: Experiment with Top-K Retrieval

In [28]:
query = "What services is the Service Provider required to perform??"
top_k_values = [2, 5, 10]

In [31]:
# A (Default)
query = "What services is the Service Provider required to perform??"
top_k_values = [2, 5, 10]

for top_k in top_k_values:
    print(f"\\n--- Results for top_k = {top_k} ---")
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query)
    for i, node in enumerate(nodes):
        print(f"Result {i+1}:")
        print(node.get_text())
        print("-" * 80)

\n--- Results for top_k = 2 ---
Result 1:
1.2 Service Provider shall use reasonable efforts to perform the Services in accordance with generally
accepted industry standards and practices.
2.
--------------------------------------------------------------------------------
Result 2:
Enterprise Town, State
67890 ("Client").
1. SERVICES
1.1 Service Provider agrees to provide Client with consulting services ("Services") as described in
Exhibit A attached hereto.
--------------------------------------------------------------------------------
\n--- Results for top_k = 5 ---
Result 1:
1.2 Service Provider shall use reasonable efforts to perform the Services in accordance with generally
accepted industry standards and practices.
2.
--------------------------------------------------------------------------------
Result 2:
Enterprise Town, State
67890 ("Client").
1. SERVICES
1.1 Service Provider agrees to provide Client with consulting services ("Services") as described in
Exhibit A attached her

In [27]:
# B
query = "What services is the Service Provider required to perform?"
top_k_values = [2, 5, 10]
threshold = 0.75

for top_k in top_k_values:
    print(f"\\n--- Results for top_k = {top_k} ---")
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query)
    for i, node in enumerate(nodes):
        print(f"Result {i+1}:")
        print(node.get_text())
        print("-" * 80)

\n--- Results for top_k = 2 ---
Result 1:
1.2 Service Provider shall use reasonable efforts to perform the Services in accordance with generally
accepted industry standards and practices.
2.
--------------------------------------------------------------------------------
Result 2:
Enterprise Town, State
67890 ("Client").
1. SERVICES
1.1 Service Provider agrees to provide Client with consulting services ("Services") as described in
Exhibit A attached hereto.
--------------------------------------------------------------------------------
\n--- Results for top_k = 5 ---
Result 1:
1.2 Service Provider shall use reasonable efforts to perform the Services in accordance with generally
accepted industry standards and practices.
2.
--------------------------------------------------------------------------------
Result 2:
Enterprise Town, State
67890 ("Client").
1. SERVICES
1.1 Service Provider agrees to provide Client with consulting services ("Services") as described in
Exhibit A attached her

In [39]:
# C-RerankerOn-Adds LLM-based reranking
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import LLMRerank
from llama_index.core import Settings
from llama_index.core.llms import LLM
from llama_index.core.base.llms.types import ChatResponse, CompletionResponse, LLMMetadata, ChatResponseGen, CompletionResponseGen
from typing import Any, Sequence, Optional

# Define a dummy LLM to prevent OpenAI API key error
# This LLM will return a fixed string and won't actually call an external API
class DummyLLM(LLM):
    @property
    def metadata(self) -> LLMMetadata:
        return LLMMetadata(model_name="dummy_model")

    def complete(self, prompt: str, **kwargs: Any) -> CompletionResponse:
        return CompletionResponse(text="This is a dummy completion response from a placeholder LLM.")

    async def acomplete(self, prompt: str, **kwargs: Any) -> CompletionResponse:
        return self.complete(prompt, **kwargs)

    def stream_complete(self, prompt: str, **kwargs: Any) -> CompletionResponseGen:
        yield CompletionResponse(text="This is a dummy streamed completion response from a placeholder LLM.")

    async def astream_complete(self, prompt: str, **kwargs: Any) -> CompletionResponseGen:
        yield self.complete(prompt, **kwargs)

    def chat(self, messages: Sequence[Any], **kwargs: Any) -> ChatResponse:
        return ChatResponse(message=messages[0]) # Just return the first message as a dummy response

    async def achat(self, messages: Sequence[Any], **kwargs: Any) -> ChatResponse:
        return self.chat(messages, **kwargs)

    def stream_chat(self, messages: Sequence[Any], **kwargs: Any) -> ChatResponseGen:
        yield ChatResponse(message=messages[0])

    async def astream_chat(self, messages: Sequence[Any], **kwargs: Any) -> ChatResponseGen:
        yield self.chat(messages, **kwargs)


# Set the dummy LLM for LlamaIndex
Settings.llm = DummyLLM()

query = "What services is the Service Provider required to perform?"

# Config C settings
top_k = 5
apply_threshold = True
threshold = 0.5  # Lowering the threshold to get some results
use_rerank = False # Disabling rerank to avoid OpenAI API key error

print("CONFIG C SETTINGS")
print("top_k =", top_k)
print("threshold ON =", apply_threshold, "| threshold =", threshold)
print("reranker ON =", use_rerank)

# 1) Retrieve
retriever = index.as_retriever(similarity_top_k=top_k)
nodes = retriever.retrieve(query)
print("\nChunks retrieved (before threshold):", len(nodes))

# 2) Threshold filter
if apply_threshold:
    nodes = [n for n in nodes if (n.score is not None and n.score >= threshold)]
print("Chunks after threshold:", len(nodes))

# 3) Print retrieved chunks (so you can log them)
if not nodes:
    print("\nNo chunks left after threshold. Try lowering threshold to 0.60 or 0.50.")
else:
    for i, node in enumerate(nodes):
        print(f"\nChunk {i+1} (Score: {node.score:.2f})")
        print(node.get_text())
        print("-" * 80)

# 4) Build query engine WITH reranker (if enabled)
postprocessors = []
if use_rerank:
    # Rerank top chunks using the LLM to improve ordering
    # Requires an LLM, e.g., OpenAI API key, or explicit LLM configuration
    postprocessors.append(LLMRerank(choice_batch_size=8, top_n=min(5, top_k)))

query_engine = RetrieverQueryEngine(
    retriever=retriever,
    node_postprocessors=postprocessors if postprocessors else None
)

# 5) Final generated answer
# Only query if there are nodes left after filtering, otherwise it might error or be meaningless
if nodes:
    response = query_engine.query(query)
    print("\nFINAL ANSWER (Config C):")
    print(response)
else:
    print("\nFINAL ANSWER (Config C): No chunks available to generate a response.")

CONFIG C SETTINGS
top_k = 5
threshold ON = True | threshold = 0.5
reranker ON = False

Chunks retrieved (before threshold): 5
Chunks after threshold: 1

Chunk 1 (Score: 0.54)
1.2 Service Provider shall use reasonable efforts to perform the Services in accordance with generally
accepted industry standards and practices.
2.
--------------------------------------------------------------------------------

FINAL ANSWER (Config C):
This is a dummy completion response from a placeholder LLM.


# ✅ STEP 3: Apply Similarity Thresholds

In [40]:
# Retrieve nodes with top_k = 10
retriever = index.as_retriever(similarity_top_k=10)
retrieved_nodes = retriever.retrieve(query)

In [41]:
# Try filtering by score
for threshold in [0.7, 0.75, 0.8]:
    filtered_nodes = [node for node in retrieved_nodes if node.score and node.score > threshold]
    print(f"\\n--- Results for threshold = {threshold} ---")
    print(f"Filtered {len(filtered_nodes)} out of {len(retrieved_nodes)} total nodes.")
    for i, node in enumerate(filtered_nodes):
        print(f"Chunk {i+1} (Score: {node.score:.2f}):")
        print(node.get_text())
        print("-" * 80)

\n--- Results for threshold = 0.7 ---
Filtered 0 out of 10 total nodes.
\n--- Results for threshold = 0.75 ---
Filtered 0 out of 10 total nodes.
\n--- Results for threshold = 0.8 ---
Filtered 0 out of 10 total nodes.


# ✅ STEP 4: Combined Configurations

In [42]:
experiments = [
    {"top_k": 5, "threshold": None},
    {"top_k": 8, "threshold": 0.75},
    {"top_k": 5, "threshold": 0.8},
]



In [43]:
for exp in experiments:
    print(f"\\n--- Experiment: top_k={exp['top_k']}, threshold={exp['threshold']} ---")
    retriever = index.as_retriever(similarity_top_k=exp["top_k"])
    nodes = retriever.retrieve(query)
    if exp["threshold"]:
        nodes = [node for node in nodes if node.score and node.score > exp["threshold"]]
    print(f"Chunks Retrieved: {len(nodes)}")
    for i, node in enumerate(nodes):
        print(f"Chunk {i+1} (Score: {node.score:.2f}):")
        print(node.get_text())
        print("-" * 80)

\n--- Experiment: top_k=5, threshold=None ---
Chunks Retrieved: 5
Chunk 1 (Score: 0.54):
1.2 Service Provider shall use reasonable efforts to perform the Services in accordance with generally
accepted industry standards and practices.
2.
--------------------------------------------------------------------------------
Chunk 2 (Score: 0.43):
Enterprise Town, State
67890 ("Client").
1. SERVICES
1.1 Service Provider agrees to provide Client with consulting services ("Services") as described in
Exhibit A attached hereto.
--------------------------------------------------------------------------------
Chunk 3 (Score: 0.40):
2. PAYMENT
2.1 Client agrees to pay Service Provider for the Services at the rates specified in Exhibit B attached
hereto.
2.2 Service Provider shall invoice Client on a monthly basis for Services performed.
--------------------------------------------------------------------------------
Chunk 4 (Score: 0.31):
, with its principal place of business at 123 Business Avenue,

✅ You can now adjust `top_k` and `threshold`, and try other queries!